# Simulation & Stylized Facts Analysis

This notebook:

1. **Part 1: Run Simulations** — Generate synthetic order-flow databases using the Hawkes-driven simulator
2. **Part 2: Stylized Facts** — Verify that simulated markets reproduce empirical stylized facts (autocorrelation, spread dynamics, mean reversion, etc.)

In [ ]:
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, gaussian_kde
from joblib import Parallel, delayed

from research_core.classes import SimulateFast, AnalyseMarket, helpers

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "axes.axisbelow": True})

In [ ]:
T = 1000000  # number of events to simulate

ARRIVAL_MODE = "hawkes_multivariate"

# Mild upward price drift via a small MO buy-bias on the kernel baseline.
# 0.0 = calibrated (near-driftless) kernel; >0 tilts price up. Calibrate
# the magnitude with the drift-calibration sweep cell further down.
DRIFT_EPS = 0.15

db_path = str(helpers.project_root() / "data" / "sim_events_10Mb.sqlite")

sim = SimulateFast(arrival_mode=ARRIVAL_MODE,
T=T,
db_path=db_path, drift_eps=DRIFT_EPS)
tick_size = 0.05  # WSE tick size for KGHM at ~98 PLN

# Load a real orderbook snapshot from WSE data
# Available assets: KGHM, PKNORLEN, PKOBP, PEKAO, PZU
# Days: d20170103 to d20171229
sim.load_real_orderbook_snapshot( 
    asset="KGHM",
    day_key="d20170110",
    snapshot_time="10:00:00",  # After opening auction
    tick_size=tick_size,
)
sim.plot_book()

In [ ]:
sim.run(overwrite=True)

## Batch Simulation Runs

Generate N independent simulation databases for downstream use by the RL pipeline notebook.

In [ ]:
t_events     = 1_000_000
arrival_mode = "hawkes_multivariate"
drift_eps    = 0.10
tick_size    = 0.05

resil_kappa      = 0.0817
resil_phi        = 0.0746
resil_tau_s      = 4.5
resil_flow_tau_s = 32.3

n_runs      = 60
max_workers = 12


def run_full_sim(run_id, t_events, arrival_mode, db_path, asset, day_key, snap_time, tick_size,
                 drift_eps=0.0, resil_kappa=0.0, resil_tau_s=10.0, resil_phi=0.0, resil_flow_tau_s=40.0):
    sim = SimulateFast(
        arrival_mode=arrival_mode, T=t_events,
        db_path=db_path, recording_mode='full', drift_eps=drift_eps,
        resil_kappa=resil_kappa, resil_tau_s=resil_tau_s,
        resil_phi=resil_phi, resil_flow_tau_s=resil_flow_tau_s,
    )
    sim.load_real_orderbook_snapshot(asset=asset, day_key=day_key, snapshot_time=snap_time, tick_size=tick_size)
    sim.run()
    return run_id


existing_dbs = sorted((helpers.project_root() / "data").glob("sim_events_1M_*.sqlite"))
if len(existing_dbs) >= n_runs:
    print(f"Found {len(existing_dbs)} existing sim databases, skipping batch run.")
else:
    args_list = [
        (i, t_events, arrival_mode,
         str(helpers.project_root() / "data" / f"sim_events_1M_{i}.sqlite"),
         "KGHM", "d20170110", "10:00:00", tick_size, drift_eps)
        for i in range(n_runs)
    ]

    t0 = time.time()
    completed = Parallel(n_jobs=max_workers, verbose=10)(
        delayed(run_full_sim)(*args,
                              resil_kappa=resil_kappa, resil_phi=resil_phi,
                              resil_tau_s=resil_tau_s, resil_flow_tau_s=resil_flow_tau_s)
        for args in args_list
    )
    print(f"Done: {len(completed)} runs in {time.time()-t0:.0f}s")

## Part 2: Stylized Facts Analysis

In [ ]:
TICK_SIZE = 0.05
SIM_RUN = 0
DATA_DIR = helpers.project_root() / "data"

db_path = DATA_DIR / f"sim_events_1M_{SIM_RUN}.sqlite"
if not db_path.exists():
    db_path = DATA_DIR / "sim_events_10Mb.sqlite"

am = AnalyseMarket(str(db_path), tick_size=TICK_SIZE)
print(f"Loaded: {db_path.name}")


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────
N_RUNS         = 30
TICK_SIZE      = 0.05
DATA_DIR       = helpers.project_root() / "data"

# Time-based resampling interval (seconds) for BBO-derived facts.
# Mid-prices are resampled onto a regular grid at this frequency
# using LOCF, and log-returns are computed from the resampled series.
RESAMPLE_DT    = 60.0  # seconds (1 minute)
DT_LABEL       = f"{RESAMPLE_DT/60:.0f}min" if RESAMPLE_DT >= 60 else f"{RESAMPLE_DT:.0f}s"

# Return ACF (lags in units of RESAMPLE_DT)
MAX_LAG_RET    = 50
# Volatility clustering ACF
MAX_LAG_VC     = 100
# Aggregational Gaussianity – multiples of RESAMPLE_DT
AGG_LEVELS     = [1, 5, 15, 30, 60, 120]

# Order sign ACF
MAX_LAG_BAR    = 50
MAX_LAG_LOGLOG = 10_000
N_LOG_BINS     = 40
FIT_CAP        = 800
# Price impact
N_BINS_IMPACT  = 20

db_paths = [DATA_DIR / f"sim_events_1M_{i}.sqlite"
            for i in range(N_RUNS)]

missing = [p for p in db_paths if not p.exists()]
if missing:
    print(f"WARNING: {len(missing)} databases not found")
else:
    print(f"All {N_RUNS} simulation databases found.")

# ═══════════════════════════════════════════════════════════════════════
# BBO extraction — single pass per database
# ═══════════════════════════════════════════════════════════════════════

FT_MINUTES = 30  # matches stylized_fat_tails(interval_minutes=30): level np.diff(sampled)
INDIVIDUAL_RUNS = [0, 1]  # 2 representative runs shown per stylized fact


def _sample_mids_locf(ts_arr, mids_arr, interval_sec):
    """LOCF mids on a regular grid — same logic as AnalyseMarket._get_sampled_mid_prices,
    but reuses already-loaded (ts, mids) so each horizon does not re-query SQLite."""
    if interval_sec <= 0 or mids_arr is None or len(mids_arr) < 5:
        return None
    ts_arr = np.asarray(ts_arr, dtype=np.float64)
    mids_arr = np.asarray(mids_arr, dtype=np.float64)
    grid = np.arange(ts_arr[0], ts_arr[-1], interval_sec)
    if len(grid) < 2:
        return None
    idx = np.clip(
        np.searchsorted(ts_arr, grid, side="right") - 1,
        0, len(mids_arr) - 1,
    )
    return mids_arr[idx]


bbo = {
    "ft_kurtoses": [], "ft_changes": [],
    "acf_ret": [], "n_rets": [],
    "acf_abs": [], "acf_sq": [], "half_lives": [],
    "agg_kurtoses": [],
}

for i, p in enumerate(db_paths):
    print(f"\r  BBO {i+1}/{N_RUNS} ...", end="", flush=True)
    _am_run = AnalyseMarket(p, tick_size=TICK_SIZE, verbose=False)
    ts, mids = _am_run._get_mid_prices()
    if mids is None or len(mids) < 5:
        bbo["n_rets"].append(0)
        bbo["ft_kurtoses"].append(np.nan)
        bbo["ft_changes"].append(np.array([]))
        bbo["acf_ret"].append(np.array([]))
        bbo["acf_abs"].append(np.array([]))
        bbo["acf_sq"].append(np.array([]))
        bbo["half_lives"].append(np.nan)
        bbo["agg_kurtoses"].append([np.nan] * len(AGG_LEVELS))
        if i in INDIVIDUAL_RUNS:
            bbo.setdefault("agg_level_returns_individual", {})[i] = {
                int(agg): np.array([]) for agg in AGG_LEVELS}
        continue

    ts = np.asarray(ts, dtype=np.float64)
    mids = np.asarray(mids, dtype=np.float64)

    # ── Resample mid-prices onto a regular 1-min grid (LOCF) ─────
    t_grid = np.arange(ts[0], ts[-1], RESAMPLE_DT)
    idx_locf = np.searchsorted(ts, t_grid, side="right") - 1
    idx_locf = np.clip(idx_locf, 0, len(mids) - 1)
    mids_resampled = mids[idx_locf]

    log_mids = np.log(mids_resampled)
    rets = np.diff(log_mids)
    n = len(rets)
    bbo["n_rets"].append(n)

    # ── Fat tails — level ΔP (same as stylized_fat_tails default / aggregational)
    sampled_ft = _sample_mids_locf(ts, mids, float(FT_MINUTES) * 60.0)
    if sampled_ft is None or len(sampled_ft) < 3:
        bbo["ft_kurtoses"].append(np.nan)
        bbo["ft_changes"].append(np.array([]))
    else:
        changes = np.diff(np.asarray(sampled_ft, dtype=np.float64))
        changes = changes[np.isfinite(changes)]
        if len(changes) < 2 or changes.std() < 1e-15:
            bbo["ft_kurtoses"].append(np.nan)
            bbo["ft_changes"].append(changes)
        else:
            mu_ft, sigma_ft = changes.mean(), changes.std()
            kurt_ft = np.mean(((changes - mu_ft) / sigma_ft) ** 4) - 3.0
            bbo["ft_kurtoses"].append(kurt_ft)
            bbo["ft_changes"].append(changes)

    # ── Return ACF ────────────────────────────────────────────────
    mu = rets.mean()
    rets_dm = rets - mu
    var = np.dot(rets_dm, rets_dm)
    lags = np.arange(1, min(MAX_LAG_RET + 1, n))
    acf_r = np.array([np.dot(rets_dm[l:], rets_dm[:-l]) / var for l in lags])
    bbo["acf_ret"].append(acf_r)

    # ── Volatility clustering ─────────────────────────────────────
    acf_a = AnalyseMarket._acf(np.abs(rets), MAX_LAG_VC)
    acf_s = AnalyseMarket._acf(rets ** 2, MAX_LAG_VC)
    bbo["acf_abs"].append(acf_a)
    bbo["acf_sq"].append(acf_s)
    if acf_a[0] > 0:
        thr = acf_a[0] / np.e
        hl_idx = np.where(acf_a < thr)[0]
        bbo["half_lives"].append(hl_idx[0] + 1 if len(hl_idx) else MAX_LAG_VC)
    else:
        bbo["half_lives"].append(np.nan)

    # ── Aggregational Gaussianity — level np.diff(sampled); horizons in minutes
    run_k = []
    indiv_agg = {}
    for agg in AGG_LEVELS:
        r = np.array([])
        sampled = _sample_mids_locf(ts, mids, float(agg) * 60.0)
        if sampled is None or len(sampled) < 25:
            run_k.append(np.nan)
        else:
            r = np.diff(np.asarray(sampled, dtype=np.float64))
            r = r[np.isfinite(r)]
            if len(r) < 20 or r.std() < 1e-15:
                run_k.append(np.nan)
            else:
                run_k.append(np.mean(((r - r.mean()) / r.std()) ** 4) - 3.0)
        if i in INDIVIDUAL_RUNS:
            indiv_agg[int(agg)] = r.copy() if r.size else np.array([])
    bbo["agg_kurtoses"].append(run_k)
    if i in INDIVIDUAL_RUNS:
        bbo.setdefault("agg_level_returns_individual", {})[i] = indiv_agg

    del ts, mids, mids_resampled, log_mids, rets

# Convert to arrays
bbo["n_rets"] = np.array(bbo["n_rets"])
bbo["ft_kurtoses"] = np.array(bbo["ft_kurtoses"])
bbo["ft_pooled"] = np.concatenate(bbo["ft_changes"])
bbo["ft_changes_individual"] = {r: bbo["ft_changes"][r] for r in INDIVIDUAL_RUNS}
del bbo["ft_changes"]
bbo["acf_ret"] = np.array(bbo["acf_ret"])
bbo["acf_abs"] = np.array(bbo["acf_abs"])
bbo["acf_sq"] = np.array(bbo["acf_sq"])
bbo["half_lives"] = np.array(bbo["half_lives"])
bbo["agg_kurtoses"] = np.array(bbo["agg_kurtoses"])

print(f"\n  Done — {len(bbo['ft_kurtoses'])} runs processed.")

# ═══════════════════════════════════════════════════════════════════════
# Mid-price paths — all 30 runs on a single plot
# ═══════════════════════════════════════════════════════════════════════

mid_series = {}
for i, p in enumerate(db_paths):
    if not p.exists():
        continue
    print(f"\r  Loading midprice {i+1}/{N_RUNS} ...", end="", flush=True)
    conn = sqlite3.connect(p)
    df = pd.read_sql("SELECT timestamp, mid_price FROM bbo ORDER BY timestamp", conn)
    conn.close()

    ts = df["timestamp"].values.astype(float)
    mids = df["mid_price"].values.astype(float)
    del df

    t_grid = np.arange(ts[0], ts[-1], RESAMPLE_DT)
    idx = np.clip(np.searchsorted(ts, t_grid, side="right") - 1, 0, len(mids) - 1)
    elapsed_min = (t_grid - ts[0]) / 60.0
    mid_series[i] = (elapsed_min, mids[idx])
    del ts, mids

print(f"\n  Done — {len(mid_series)} runs loaded.")

fig, ax = plt.subplots(figsize=(14, 6))
cmap = plt.cm.viridis(np.linspace(0, 1, N_RUNS))
for i, (t, m) in mid_series.items():
    ax.plot(t / 60, m, lw=0.5, alpha=0.7, color=cmap[i])
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Mid price")
ax.set_title(f"Mid-price paths — {len(mid_series)} simulation runs")
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# 1. Fat tails  (30-min level price changes — matches stylized_fat_tails / empirical)
# ═══════════════════════════════════════════════════════════════════════

pooled = bbo["ft_pooled"]
k = bbo["ft_kurtoses"]
mu_p, sigma_p = pooled.mean(), pooled.std()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── Row 1: aggregated 30-run view ──────────────────────────────────
ax = axes[0, 0]
kde = gaussian_kde(pooled)
x = np.linspace(pooled.min(), pooled.max(), 500)
ax.plot(x, norm.pdf(x, mu_p, sigma_p), "b-", lw=2, label="Gaussian")
ax.plot(x, kde(x), "r-", lw=2, label="30 min level ΔP (pooled)")
ax.set_xlabel("Price change"); ax.set_ylabel("Density")
ax.set_title("Density of 30 min level price changes"); ax.legend()

ax = axes[0, 1]
sr = np.sort(pooled)
step_qq = max(1, len(sr) // 5000)
sr_s = sr[::step_qq]; n_s = len(sr_s)
tq = np.array([mu_p + sigma_p * AnalyseMarket._ppf_normal((i + 0.5) / n_s)
               for i in range(n_s)])
ax.scatter(tq, sr_s, s=1, alpha=0.4)
lims = [min(tq.min(), sr_s.min()), max(tq.max(), sr_s.max())]
ax.plot(lims, lims, "r--", lw=1, label="45° line")
ax.set_xlabel("Normal theoretical quantiles")
ax.set_ylabel("Sample quantiles")
ax.set_title(f"QQ-plot (pooled, {N_RUNS} runs)"); ax.legend()

ax = axes[0, 2]
ax.bar(range(len(k)), k, alpha=0.7, color="steelblue")
ax.axhline(k.mean(), color="crimson", lw=2, ls="--",
           label=f"Mean = {k.mean():.0f}")
ax.set_yscale("log")
ax.set_xlabel("Run"); ax.set_ylabel("Excess kurtosis")
ax.set_title("Per-run excess kurtosis"); ax.legend()

# ── Row 2: individual run analysis (2 runs) ────────────────────────
for col, run_idx in enumerate(INDIVIDUAL_RUNS):
    changes = bbo["ft_changes_individual"][run_idx]
    mu_i, sig_i = changes.mean(), changes.std()
    kurt_i = bbo["ft_kurtoses"][run_idx]

    ax = axes[1, col]
    kde_i = gaussian_kde(changes)
    x_i = np.linspace(changes.min(), changes.max(), 500)
    ax.plot(x_i, norm.pdf(x_i, mu_i, sig_i), "b-", lw=2, label="Gaussian")
    ax.plot(x_i, kde_i(x_i), "r-", lw=2, label=f"Run {run_idx}")
    ax.set_xlabel("Price change"); ax.set_ylabel("Density")
    ax.set_title(f"Run {run_idx} — KDE (κ = {kurt_i:.1f})"); ax.legend()

# hide unused subplot in row 2
axes[1, 2].set_visible(False)

plt.suptitle(f"Fat tails (30 min level ΔP, AnalyseMarket sampling) — Mean κ = {k.mean():.0f} ± {k.std():.0f}  "
             f"({len(k)} runs)", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print(f"Per-run excess kurtosis: {k.mean():.0f} ± {k.std():.0f}  "
      f"(min={k.min():.0f}, max={k.max():.0f})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2. Absence of return autocorrelation
# ═══════════════════════════════════════════════════════════════════════

acf_ret = bbo["acf_ret"]
mean_acf = acf_ret.mean(axis=0)
std_acf = acf_ret.std(axis=0)
lags = np.arange(1, len(mean_acf) + 1)
ci = 1.96 / np.sqrt(bbo["n_rets"].mean())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Panel 1: aggregated mean ± std ─────────────────────────────────
ax = axes[0]
ax.bar(lags, mean_acf, width=0.8, alpha=0.7, color="steelblue",
       yerr=std_acf, capsize=1, ecolor="grey", label="Mean ± std")
ax.axhline(ci,  ls="--", color="red", alpha=0.5, label="95% CI")
ax.axhline(-ci, ls="--", color="red", alpha=0.5)
ax.axhline(0, color="black", lw=0.5)
ax.set_xlabel(f"Lag ({DT_LABEL})"); ax.set_ylabel("Autocorrelation")
ax.set_title(f"ACF of {DT_LABEL} log-returns ({len(acf_ret)} runs)")
ax.legend(fontsize=9)

# ── Panels 2-3: individual runs ────────────────────────────────────
for col, run_idx in enumerate(INDIVIDUAL_RUNS):
    ax = axes[1 + col]
    ci_i = 1.96 / np.sqrt(bbo["n_rets"][run_idx])
    ax.bar(lags, acf_ret[run_idx], width=0.8, alpha=0.7, color="steelblue")
    ax.axhline(ci_i,  ls="--", color="red", alpha=0.5, label="95% CI")
    ax.axhline(-ci_i, ls="--", color="red", alpha=0.5)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel(f"Lag ({DT_LABEL})"); ax.set_ylabel("Autocorrelation")
    ax.set_title(f"Run {run_idx}"); ax.legend(fontsize=9)

plt.suptitle("Absence of return autocorrelation", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

mean_abs_acf = np.array([np.abs(a).mean() for a in acf_ret])
print(f"Mean |ACF| across lags 1–{MAX_LAG_RET}: "
      f"{mean_abs_acf.mean():.6f} ± {mean_abs_acf.std():.6f}")

# ═══════════════════════════════════════════════════════════════════════
# 3. Volatility clustering (same layout as empirical: linear axes, bar ACF)
# ═══════════════════════════════════════════════════════════════════════

mean_abs = bbo["acf_abs"].mean(axis=0)
std_abs  = bbo["acf_abs"].std(axis=0)
mean_sq  = bbo["acf_sq"].mean(axis=0)
std_sq   = bbo["acf_sq"].std(axis=0)

interval_min = RESAMPLE_DT / 60.0  # minutes per lag (matches empirical interval_minutes)
lag_minutes = np.arange(1, len(mean_abs) + 1) * interval_min
bar_w = interval_min * 0.8
ci_mean_n = 1.96 / np.sqrt(bbo["n_rets"].mean())

n_ind = len(INDIVIDUAL_RUNS)
fig, axes = plt.subplots(1 + n_ind, 2, figsize=(13, 5 + 3.5 * n_ind), sharey=True)

# ── Row 0: mean ACF across runs (bars + asymptotic band, like empirical) ──
ax = axes[0, 0]
ax.bar(lag_minutes, mean_abs, width=bar_w, alpha=0.7, color="tab:blue")
ax.axhline(ci_mean_n, ls="--", color="red", alpha=0.5)
ax.axhline(-ci_mean_n, ls="--", color="red", alpha=0.5)
ax.axhline(0, color="black", lw=0.5)
ax.set_xlabel("Lag (minutes)")
ax.set_ylabel("Autocorrelation")
ax.set_title("ACF of |returns|")

ax = axes[0, 1]
ax.bar(lag_minutes, mean_sq, width=bar_w, alpha=0.7, color="tab:orange")
ax.axhline(ci_mean_n, ls="--", color="red", alpha=0.5)
ax.axhline(-ci_mean_n, ls="--", color="red", alpha=0.5)
ax.axhline(0, color="black", lw=0.5)
ax.set_xlabel("Lag (minutes)")
ax.set_title("ACF of returns²")

# ── Rows 1+: individual runs ─────────────────────────────────────────
for row, run_idx in enumerate(INDIVIDUAL_RUNS):
    ci_i = 1.96 / np.sqrt(bbo["n_rets"][run_idx])
    ax = axes[1 + row, 0]
    ax.bar(lag_minutes, bbo["acf_abs"][run_idx], width=bar_w, alpha=0.7, color="tab:blue")
    ax.axhline(ci_i, ls="--", color="red", alpha=0.5)
    ax.axhline(-ci_i, ls="--", color="red", alpha=0.5)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel("Lag (minutes)")
    ax.set_ylabel("Autocorrelation")
    ax.set_title(f"Run {run_idx} — ACF of |returns|")

    ax = axes[1 + row, 1]
    ax.bar(lag_minutes, bbo["acf_sq"][run_idx], width=bar_w, alpha=0.7, color="tab:orange")
    ax.axhline(ci_i, ls="--", color="red", alpha=0.5)
    ax.axhline(-ci_i, ls="--", color="red", alpha=0.5)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel("Lag (minutes)")
    ax.set_title(f"Run {run_idx} — ACF of returns²")

hl = bbo["half_lives"]
plt.suptitle(
    "Volatility clustering  (slow decay → clustering present)\n"
    f"({len(bbo['acf_abs'])} runs; row 0 = mean ACF)",
    fontsize=13,
)
plt.tight_layout(); plt.show()

print(f"|returns| ACF half-life: {hl.mean():.1f} ± {hl.std():.1f} {DT_LABEL}  "
      f"(min={hl.min():.0f}, max={hl.max():.0f} {DT_LABEL})")
print(f"|returns| ACF(1):  {mean_abs[0]:.4f} ± {std_abs[0]:.4f}")
print(f"|returns| ACF(10): {mean_abs[9]:.4f} ± {std_abs[9]:.4f}")

In [ ]:
db_path_single = str(helpers.project_root() / "data" / "sim_events_1M_0.sqlite")
am_single = AnalyseMarket(db_path_single, tick_size=TICK_SIZE)
am_single.plot_candlestick(timeframe=60)
am_single.plot_depth(n_events=1_000_000, offset=0)

In [ ]:
# TODO: Resiliency / normalized signature plot (VR(τ) empirical vs sim).
# Requires running the full resiliency calibration pipeline first
# (see notebooks/resiliency_calibration.ipynb for the complete workflow).
# Key dependencies: load_empirical_vr(), build_template(), run_batch_4d(), vr_stats()
pass

In [ ]:
# TODO: Aggressor markout / mean-reversion plot.
# Requires pre-computed `sources` dict with MO timestamps and signed mid moves
# (see notebooks/mean_reversion_evidence copy.ipynb for the full pipeline).
# Key dependencies: markout_curve(), sources dict, col dict
pass